In [0]:
# Load data
base_path = "/Volumes/workspace/default/phase5/"

customers = spark.read.csv(f"{base_path}olist_customers_dataset.csv", header=True, inferSchema=True)
geolocation = spark.read.csv(f"{base_path}olist_geolocation_dataset.csv", header=True, inferSchema=True)
order_items = spark.read.csv(f"{base_path}olist_order_items_dataset.csv", header=True, inferSchema=True)
payments = spark.read.csv(f"{base_path}olist_order_payments_dataset.csv", header=True, inferSchema=True)
reviews = spark.read.csv(f"{base_path}olist_order_reviews_dataset.csv", header=True, inferSchema=True)
orders = spark.read.csv(f"{base_path}olist_orders_dataset.csv", header=True, inferSchema=True)
products = spark.read.csv(f"{base_path}olist_products_dataset.csv", header=True, inferSchema=True)
sellers = spark.read.csv(f"{base_path}olist_sellers_dataset.csv", header=True, inferSchema=True)
category_translation = spark.read.csv(f"{base_path}product_category_name_translation.csv", header=True, inferSchema=True)

In [0]:
# Import functions
from pyspark.sql.functions import col, sum, count, countDistinct, avg, max, min, when, to_date, date_format, month, year, datediff, rank, dense_rank, row_number, lag, lead, round, desc, asc
from pyspark.sql.window import Window

In [0]:
# Check schema
customers.printSchema()
display(customers.limit(3))

orders.printSchema()
display(orders.limit(3))

order_items.printSchema()
display(order_items.limit(3))

products.printSchema()
display(products.limit(3))

root
 |-- customer_id: string (nullable = true)
 |-- customer_unique_id: string (nullable = true)
 |-- customer_zip_code_prefix: integer (nullable = true)
 |-- customer_city: string (nullable = true)
 |-- customer_state: string (nullable = true)



customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,franca,SP
18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,9790,sao bernardo do campo,SP
4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,1151,sao paulo,SP


root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- order_purchase_timestamp: timestamp (nullable = true)
 |-- order_approved_at: timestamp (nullable = true)
 |-- order_delivered_carrier_date: timestamp (nullable = true)
 |-- order_delivered_customer_date: timestamp (nullable = true)
 |-- order_estimated_delivery_date: timestamp (nullable = true)



order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02T10:56:33.000Z,2017-10-02T11:07:15.000Z,2017-10-04T19:55:00.000Z,2017-10-10T21:25:13.000Z,2017-10-18T00:00:00.000Z
53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24T20:41:37.000Z,2018-07-26T03:24:27.000Z,2018-07-26T14:31:00.000Z,2018-08-07T15:27:45.000Z,2018-08-13T00:00:00.000Z
47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08T08:38:49.000Z,2018-08-08T08:55:23.000Z,2018-08-08T13:50:00.000Z,2018-08-17T18:06:29.000Z,2018-09-04T00:00:00.000Z


root
 |-- order_id: string (nullable = true)
 |-- order_item_id: integer (nullable = true)
 |-- product_id: string (nullable = true)
 |-- seller_id: string (nullable = true)
 |-- shipping_limit_date: timestamp (nullable = true)
 |-- price: double (nullable = true)
 |-- freight_value: double (nullable = true)



order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19T09:45:35.000Z,58.9,13.29
00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03T11:05:13.000Z,239.9,19.93
000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18T14:48:30.000Z,199.0,17.87


root
 |-- product_id: string (nullable = true)
 |-- product_category_name: string (nullable = true)
 |-- product_name_lenght: integer (nullable = true)
 |-- product_description_lenght: integer (nullable = true)
 |-- product_photos_qty: integer (nullable = true)
 |-- product_weight_g: integer (nullable = true)
 |-- product_length_cm: integer (nullable = true)
 |-- product_height_cm: integer (nullable = true)
 |-- product_width_cm: integer (nullable = true)



product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
1e9e8ef04dbcff4541ed26657ea517e5,perfumaria,40,287,1,225,16,10,14
3aa071139cb16b67ca9e5dea641aaa2f,artes,44,276,1,1000,30,18,20
96bd76ec8810374ed1b65e291975717f,esporte_lazer,46,250,1,154,18,9,15


In [0]:
# Practice Set A: Join Drills

# Inner join
inner_join_result = orders.join(customers, "customer_id", "inner")
display(inner_join_result.select("order_id", "customer_id", "customer_city", "order_status").limit(10))

# Left join
left_join_result = orders.join(customers, "customer_id", "left")
display(left_join_result.select("order_id", "customer_id", "customer_city", "order_status").limit(10))

# Left anti join
left_anti_result = orders.join(customers, "customer_id", "left_anti")
print(f"Orphaned orders: {left_anti_result.count()}")

# Check referential integrity
order_items_orphaned = order_items.join(orders, "order_id", "left_anti").count()
products_orphaned = order_items.join(products, "product_id", "left_anti").count()
print(f"Order items without orders: {order_items_orphaned}")
print(f"Order items without products: {products_orphaned}")

order_id,customer_id,customer_city,order_status
e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,sao paulo,delivered
53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,barreiras,delivered
47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,vianopolis,delivered
949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,sao goncalo do amarante,delivered
ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,santo andre,delivered
a4591c265e18cb1dcee52889e2d8acc3,503740e9ca751ccdda7ba28e9ab8f608,congonhinhas,delivered
136cce7faa42fdb2cefd53fdc79a6098,ed0271e0b7da060a393796590e7b737a,santa rosa,invoiced
6514b8ad8028c9f2cc2374ded245783f,9bdf08b4b3b52b5526ff42d37d47f222,nilopolis,delivered
76c6e866289321a7c93b82b54852dc33,f54a9f0e6b351c431402b8461ea51999,faxinalzinho,delivered
e69bfb5eb88e0ed6a785585b27e16dbf,31ad1d1b63eb9962463f764d4e6e0c9d,sorocaba,delivered


order_id,customer_id,customer_city,order_status
e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,sao paulo,delivered
53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,barreiras,delivered
47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,vianopolis,delivered
949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,sao goncalo do amarante,delivered
ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,santo andre,delivered
a4591c265e18cb1dcee52889e2d8acc3,503740e9ca751ccdda7ba28e9ab8f608,congonhinhas,delivered
136cce7faa42fdb2cefd53fdc79a6098,ed0271e0b7da060a393796590e7b737a,santa rosa,invoiced
6514b8ad8028c9f2cc2374ded245783f,9bdf08b4b3b52b5526ff42d37d47f222,nilopolis,delivered
76c6e866289321a7c93b82b54852dc33,f54a9f0e6b351c431402b8461ea51999,faxinalzinho,delivered
e69bfb5eb88e0ed6a785585b27e16dbf,31ad1d1b63eb9962463f764d4e6e0c9d,sorocaba,delivered


Orphaned orders: 0
Order items without orders: 0
Order items without products: 0


In [0]:
# Practice Set B: Window Functions

# Prepare data
customer_orders = orders.join(order_items, "order_id").join(customers, "customer_id").select("customer_id", "customer_city", "customer_state", "order_id", "price")

# Customer total spending
customer_spending = customer_orders.groupBy("customer_id", "customer_city", "customer_state").agg(sum("price").alias("total_spent"), count("order_id").alias("order_count"))
display(customer_spending.orderBy(col("total_spent").desc()).limit(10))

# Global ranking
window_global = Window.orderBy(col("total_spent").desc())
customer_ranked = customer_spending.withColumn("global_rank", rank().over(window_global))
display(customer_ranked.filter(col("global_rank") <= 10).orderBy("global_rank"))

# City level ranking
window_city = Window.partitionBy("customer_city").orderBy(col("total_spent").desc())
customer_city_ranked = customer_spending.withColumn("city_rank", rank().over(window_city)).filter(col("city_rank") <= 3)
display(customer_city_ranked.limit(30))

# Running totals
window_running = Window.partitionBy("customer_state").orderBy(col("total_spent").desc()).rowsBetween(Window.unboundedPreceding, Window.currentRow)
customer_running_total = customer_spending.withColumn("running_total", sum("total_spent").over(window_running))
display(customer_running_total.limit(20))

# LAG and LEAD
window_sequence = Window.partitionBy("customer_state").orderBy(col("total_spent").desc())
customer_comparison = customer_spending.withColumn("prev_customer_spent", lag("total_spent").over(window_sequence)).withColumn("next_customer_spent", lead("total_spent").over(window_sequence))
display(customer_comparison.limit(20))

customer_id,customer_city,customer_state,total_spent,order_count
1617b1357756262bfa56ab541c47bc16,rio de janeiro,RJ,13440.0,8
ec5b2ba62e574342386871631fafd3fc,vila velha,ES,7160.0,4
c6e2731c5b391845f6800c97401a43a9,campo grande,MS,6735.0,1
f48d464a0baaea338cb25f816991ab1f,vitoria,ES,6729.0,1
3fd6777bbce08a352fddd04e4a7cc8f6,marilia,SP,6499.0,1
05455dfa7cd02f13d132aa7a6a9729c6,divinopolis,MG,5934.6,6
df55c14d1476a9a3467f131269c2477f,araruama,RJ,4799.0,1
24bbf5fd2f2e1b359ee7de94defc4a15,maua,SP,4690.0,1
e0a2412720e9ea4f26c1ac985f6a7358,goiania,GO,4599.9,2
3d979689f636322c62418b6346b1c6d2,joao pessoa,PB,4590.0,1


/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


customer_id,customer_city,customer_state,total_spent,order_count,global_rank
1617b1357756262bfa56ab541c47bc16,rio de janeiro,RJ,13440.0,8,1
ec5b2ba62e574342386871631fafd3fc,vila velha,ES,7160.0,4,2
c6e2731c5b391845f6800c97401a43a9,campo grande,MS,6735.0,1,3
f48d464a0baaea338cb25f816991ab1f,vitoria,ES,6729.0,1,4
3fd6777bbce08a352fddd04e4a7cc8f6,marilia,SP,6499.0,1,5
05455dfa7cd02f13d132aa7a6a9729c6,divinopolis,MG,5934.6,6,6
df55c14d1476a9a3467f131269c2477f,araruama,RJ,4799.0,1,7
24bbf5fd2f2e1b359ee7de94defc4a15,maua,SP,4690.0,1,8
e0a2412720e9ea4f26c1ac985f6a7358,goiania,GO,4599.9,2,9
3d979689f636322c62418b6346b1c6d2,joao pessoa,PB,4590.0,1,10


customer_id,customer_city,customer_state,total_spent,order_count,city_rank
9e01f714a2b3b8962c222cf2b74c20dc,abadia dos dourados,MG,199.0,1,1
a23e3f9a2b656b23b7e52075964b42cd,abadia dos dourados,MG,120.0,1,2
f11eb8f0b8b87510a93e3e1aa10b0ade,abadia dos dourados,MG,39.9,1,3
576d71ddb21b21763cfedce73b902180,abadiania,GO,949.99,1,1
d47c8bb51df6f716e196ecd6cd5d2c09,abaete,MG,449.0,1,1
ff0d62f8be4c098e6306f39bc6ebded4,abaete,MG,225.9,1,2
5371894984937a27cf40c7d20699a786,abaete,MG,208.9,1,3
c7eb06383ae604616cb4c9d36fe6745e,abaetetuba,PA,1500.0,1,1
367fd22f1e994de87cf447d29c167a1f,abaetetuba,PA,797.6,1,2
7727e2cc9ec428ad3173cc812ce1781b,abaetetuba,PA,580.27,1,3


customer_id,customer_city,customer_state,total_spent,order_count,running_total
711fff4266b53bae9de25be1473dc0bc,rio branco,AC,1200.0,1,1200.0
cd281c1a7d26cd29a3ed4b029fce7270,rio branco,AC,961.6,1,2161.6
f23c4b530f6d7d421de1e38d3e0cc327,cruzeiro do sul,AC,839.99,1,3001.59
3743de9608dba0325a5534fff7c367d6,rio branco,AC,809.1,1,3810.69
714f3ff34cd22b97f37d42bdef4bbc88,rio branco,AC,549.0,1,4359.6900000000005
8ff83690a72dc5589bc39742798a128b,rio branco,AC,527.9,1,4887.59
4551d3400f68fda36d0062727b26c04b,rio branco,AC,524.9,1,5412.49
5da179829d2103e0bd3f9fb2add6cb93,rio branco,AC,510.0,1,5922.49
30e7b476534296021a9f7e0c289c6a86,rio branco,AC,479.4,6,6401.889999999999
c5de7db594b246f428940ced283125bd,rio branco,AC,399.0,1,6800.889999999999


customer_id,customer_city,customer_state,total_spent,order_count,prev_customer_spent,next_customer_spent
711fff4266b53bae9de25be1473dc0bc,rio branco,AC,1200.0,1,null,961.6
cd281c1a7d26cd29a3ed4b029fce7270,rio branco,AC,961.6,1,1200.0,839.99
f23c4b530f6d7d421de1e38d3e0cc327,cruzeiro do sul,AC,839.99,1,961.6,809.1
3743de9608dba0325a5534fff7c367d6,rio branco,AC,809.1,1,839.99,549.0
714f3ff34cd22b97f37d42bdef4bbc88,rio branco,AC,549.0,1,809.1,527.9
8ff83690a72dc5589bc39742798a128b,rio branco,AC,527.9,1,549.0,524.9
4551d3400f68fda36d0062727b26c04b,rio branco,AC,524.9,1,527.9,510.0
5da179829d2103e0bd3f9fb2add6cb93,rio branco,AC,510.0,1,524.9,479.4
30e7b476534296021a9f7e0c289c6a86,rio branco,AC,479.4,6,510.0,399.0
c5de7db594b246f428940ced283125bd,rio branco,AC,399.0,1,479.4,388.0


In [0]:
# Practice Set C: Date Analysis

# Prepare data
orders_with_dates = orders.join(order_items, "order_id").select("order_id", "customer_id", "order_purchase_timestamp", "order_delivered_customer_date", "order_estimated_delivery_date", "price")

# Extract date parts
orders_with_month = orders_with_dates.withColumn("purchase_date", to_date("order_purchase_timestamp")).withColumn("month", month("purchase_date")).withColumn("year", year("purchase_date")).withColumn("year_month", date_format("purchase_date", "yyyy-MM"))
display(orders_with_month.select("order_id", "purchase_date", "year", "month", "year_month", "price").limit(10))

# Monthly sales
monthly_sales = orders_with_month.groupBy("year", "month", "year_month").agg(sum("price").alias("total_sales"), count("order_id").alias("order_count"), countDistinct("customer_id").alias("unique_customers")).orderBy("year", "month")
display(monthly_sales)

# Delivery time analysis
delivery_analysis = orders.filter(col("order_delivered_customer_date").isNotNull() & col("order_estimated_delivery_date").isNotNull()).withColumn("actual_delivery_days", datediff(to_date("order_delivered_customer_date"), to_date("order_purchase_timestamp"))).withColumn("estimated_delivery_days", datediff(to_date("order_estimated_delivery_date"), to_date("order_purchase_timestamp"))).withColumn("delivery_delay_days", datediff(to_date("order_delivered_customer_date"), to_date("order_estimated_delivery_date"))).withColumn("on_time", when(col("delivery_delay_days") <= 0, "Yes").otherwise("No"))
display(delivery_analysis.select("order_id", "order_status", "actual_delivery_days", "estimated_delivery_days", "delivery_delay_days", "on_time").limit(20))

# Month over month growth
window_mom = Window.orderBy("year", "month")
monthly_trends = monthly_sales.withColumn("prev_month_sales", lag("total_sales").over(window_mom)).withColumn("mom_growth_pct", round(((col("total_sales") - col("prev_month_sales")) / col("prev_month_sales")) * 100, 2))
display(monthly_trends.select("year_month", "total_sales", "prev_month_sales", "mom_growth_pct"))

order_id,purchase_date,year,month,year_month,price
e481f51cbdc54678b7cc49136f2d6af7,2017-10-02,2017,10,2017-10,29.99
53cdb2fc8bc7dce0b6741e2150273451,2018-07-24,2018,7,2018-07,118.7
47770eb9100c2d0c44946d9cf07ec65d,2018-08-08,2018,8,2018-08,159.9
949d5b44dbf5de918fe9c16f97b45f8a,2017-11-18,2017,11,2017-11,45.0
ad21c59c0840e6cb83a9ceb5573f8159,2018-02-13,2018,2,2018-02,19.9
a4591c265e18cb1dcee52889e2d8acc3,2017-07-09,2017,7,2017-07,147.9
136cce7faa42fdb2cefd53fdc79a6098,2017-04-11,2017,4,2017-04,49.9
6514b8ad8028c9f2cc2374ded245783f,2017-05-16,2017,5,2017-05,59.99
76c6e866289321a7c93b82b54852dc33,2017-01-23,2017,1,2017-01,19.9
e69bfb5eb88e0ed6a785585b27e16dbf,2017-07-29,2017,7,2017-07,149.99


year,month,year_month,total_sales,order_count,unique_customers
2016,9,2016-09,267.36,6,3
2016,10,2016-10,49507.6600000001,363,308
2016,12,2016-12,10.9,1,1
2017,1,2017-01,120312.86999999976,955,789
2017,2,2017-02,247303.01999999626,1951,1733
2017,3,2017-03,374344.3000000024,3000,2641
2017,4,2017-04,359927.2300000004,2684,2391
2017,5,2017-05,506071.1400000057,4136,3660
2017,6,2017-06,433038.6000000034,3583,3217
2017,7,2017-07,498031.4800000085,4519,3969


order_id,order_status,actual_delivery_days,estimated_delivery_days,delivery_delay_days,on_time
e481f51cbdc54678b7cc49136f2d6af7,delivered,8,16,-8,Yes
53cdb2fc8bc7dce0b6741e2150273451,delivered,14,20,-6,Yes
47770eb9100c2d0c44946d9cf07ec65d,delivered,9,27,-18,Yes
949d5b44dbf5de918fe9c16f97b45f8a,delivered,14,27,-13,Yes
ad21c59c0840e6cb83a9ceb5573f8159,delivered,3,13,-10,Yes
a4591c265e18cb1dcee52889e2d8acc3,delivered,17,23,-6,Yes
6514b8ad8028c9f2cc2374ded245783f,delivered,10,22,-12,Yes
76c6e866289321a7c93b82b54852dc33,delivered,10,42,-32,Yes
e69bfb5eb88e0ed6a785585b27e16dbf,delivered,18,25,-7,Yes
e6ce16cb79ec1d90b1da9085a6118aeb,delivered,13,22,-9,Yes


/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


year_month,total_sales,prev_month_sales,mom_growth_pct
2016-09,267.36,null,null
2016-10,49507.659999999974,267.36,18417.23
2016-12,10.9,49507.659999999974,-99.98
2017-01,120312.86999999992,10.9,1103687.8
2017-02,247303.02000000048,120312.86999999992,105.55
2017-03,374344.3000000009,247303.02000000048,51.37
2017-04,359927.2300000008,374344.3000000009,-3.85
2017-05,506071.1400000008,359927.2300000008,40.6
2017-06,433038.60000000114,506071.1400000008,-14.43
2017-07,498031.48000000103,433038.60000000114,15.01


In [0]:
# Practice Set D: Complete Pipeline

# Clean data
clean_orders = orders.filter(col("customer_id").isNotNull() & col("order_purchase_timestamp").isNotNull())
clean_order_items = order_items.filter(col("order_id").isNotNull() & col("product_id").isNotNull() & col("price").isNotNull() & (col("price") > 0))
clean_products = products.filter(col("product_category_name").isNotNull())

# Join tables
master_data = clean_order_items.join(clean_orders, "order_id").join(customers, "customer_id").join(clean_products, "product_id").join(sellers, "seller_id").select("order_id", "order_purchase_timestamp", "order_status", "customer_id", col("customer_city").alias("city"), col("customer_state").alias("state"), "product_id", "product_category_name", "seller_id", "price", "freight_value")

display(master_data.limit(5))

# Customer metrics
customer_metrics = master_data.groupBy("customer_id", "city", "state").agg(sum("price").alias("total_spend"), count("order_id").alias("order_count"), countDistinct("product_category_name").alias("unique_categories")).withColumn("segment", when(col("total_spend") > 10000, "Gold").when(col("total_spend") >= 5000, "Silver").otherwise("Bronze"))

window_global = Window.orderBy(col("total_spend").desc())
customer_final = customer_metrics.withColumn("global_rank", rank().over(window_global))

display(customer_final.orderBy("global_rank").limit(20))

# Category performance
category_performance = master_data.groupBy("product_category_name").agg(sum("price").alias("total_revenue"), count("order_id").alias("order_count"), countDistinct("customer_id").alias("unique_customers")).orderBy(col("total_revenue").desc())

display(category_performance.limit(15))

# Monthly sales trends
monthly_summary = master_data.withColumn("purchase_date", to_date("order_purchase_timestamp")).withColumn("year_month", date_format("purchase_date", "yyyy-MM")).groupBy("year_month").agg(sum("price").alias("monthly_revenue"), count("order_id").alias("order_count"), countDistinct("customer_id").alias("unique_customers")).orderBy("year_month")

window_time = Window.orderBy("year_month").rowsBetween(Window.unboundedPreceding, Window.currentRow)
monthly_final = monthly_summary.withColumn("cumulative_revenue", sum("monthly_revenue").over(window_time)).withColumn("prev_month_revenue", lag("monthly_revenue").over(Window.orderBy("year_month"))).withColumn("mom_growth_pct", round(((col("monthly_revenue") - col("prev_month_revenue")) / col("prev_month_revenue")) * 100, 2))

display(monthly_final)

order_id,order_purchase_timestamp,order_status,customer_id,city,state,product_id,product_category_name,seller_id,price,freight_value
00125cb692d04887809806618a2a145f,2017-03-23T12:21:17.000Z,delivered,8afb90a97ee661103014329b1bcea1a2,anapolis,GO,1c0c0093a48f13ba70d0c6b0a9157cb7,moveis_decoracao,41b39e28db005d9731d9d485a83b4c38,109.9,25.51
004eab0fd8f28adaf8d488976f77febe,2017-08-02T15:32:46.000Z,delivered,c476ddfbabfa4624603e0b7f8e245057,rio de janeiro,RJ,0c753fe6a58dfa7a27bda5de76e779c3,esporte_lazer,7586919161935337bf6b6d7ff5779648,21.5,11.98
00995d799817ecc3bd2abd8fbe59c430,2018-07-26T11:25:18.000Z,delivered,266636baec3cfa8d854713f0b465a2cb,sao goncalo,RJ,f3b8bfa5b86249e75e5c0632acc0e82e,beleza_saude,620c87c171fb2a6dd6e8bb4dec959fc6,49.9,12.92
00bdcdda88e6b02977fc6ce3d412c600,2018-06-12T09:51:55.000Z,delivered,45ba03e2c6bbb5dc48131ba32ec3ae5e,rio de janeiro,RJ,20566af10fda4783935b8cef249fa4c3,relogios_presentes,d921b68bf747894be13a97ae52b0f386,118.9,18.93
0115d160c5fbd758a139ff90821db60d,2017-08-29T19:14:43.000Z,delivered,9e593469b6f1beac77c2da4d73bf1228,patos de minas,MG,766a6a4c51c63991e78e192d802787a9,construcao_ferramentas_construcao,59b22a78efb79a4797979612b885db36,389.0,22.63


/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


customer_id,city,state,total_spend,order_count,unique_categories,segment,global_rank
1617b1357756262bfa56ab541c47bc16,rio de janeiro,RJ,13440.0,8,1,Gold,1
ec5b2ba62e574342386871631fafd3fc,vila velha,ES,7160.0,4,1,Silver,2
c6e2731c5b391845f6800c97401a43a9,campo grande,MS,6735.0,1,1,Silver,3
f48d464a0baaea338cb25f816991ab1f,vitoria,ES,6729.0,1,1,Silver,4
3fd6777bbce08a352fddd04e4a7cc8f6,marilia,SP,6499.0,1,1,Silver,5
05455dfa7cd02f13d132aa7a6a9729c6,divinopolis,MG,5934.6,6,1,Silver,6
df55c14d1476a9a3467f131269c2477f,araruama,RJ,4799.0,1,1,Bronze,7
24bbf5fd2f2e1b359ee7de94defc4a15,maua,SP,4690.0,1,1,Bronze,8
e0a2412720e9ea4f26c1ac985f6a7358,goiania,GO,4599.9,2,1,Bronze,9
3d979689f636322c62418b6346b1c6d2,joao pessoa,PB,4590.0,1,1,Bronze,10


product_category_name,total_revenue,order_count,unique_customers
beleza_saude,1258681.3399999766,9670,8836
relogios_presentes,1205005.6799999971,5991,5624
cama_mesa_banho,1036988.6800000572,11115,9417
esporte_lazer,988048.9700000364,8641,7720
informatica_acessorios,911954.3200000301,7827,6689
moveis_decoracao,729762.4900000283,8334,6449
cool_stuff,635290.8500000002,3796,3632
utilidades_domesticas,632248.6600000157,6964,5884
automotivo,592720.110000009,4235,3897
ferramentas_jardim,485256.4600000066,4347,3518


year_month,monthly_revenue,order_count,unique_customers,cumulative_revenue,prev_month_revenue,mom_growth_pct
2016-09,267.36,6,3,267.36,null,null
2016-10,49441.77000000013,361,306,49709.13000000013,267.36,18392.58
2016-12,10.9,1,1,49720.03000000013,49441.77000000013,-99.98
2017-01,118610.22999999968,942,778,168330.2599999998,10.9,1088067.25
2017-02,238268.91999999635,1893,1681,406599.17999999615,118610.22999999968,100.88
2017-03,367630.12999999983,2937,2589,774229.309999996,238268.91999999635,54.29
2017-04,351234.14000000025,2613,2329,1125463.4499999962,367630.12999999983,-4.46
2017-05,495109.20000000624,4040,3577,1620572.6500000025,351234.14000000025,40.96
2017-06,427510.8100000023,3520,3163,2048083.4600000049,495109.20000000624,-13.65
2017-07,494906.8200000064,4458,3918,2542990.2800000114,427510.8100000023,15.76
